# PP-OCRv6 + VLM Correction Pipeline — Colab Runner

Notebook này hướng dẫn chạy pipeline OCR + VLM correction **từng bước** trên Google Colab.

**Yêu cầu:** Chọn Runtime → Change runtime type → **GPU (T4)** trước khi bắt đầu.

---

## Tổng quan các bước

| Bước | Script | Mô tả | GPU? |
|------|--------|-------|------|
| 1 | `01_build_frame_registry.py` | Quét frames, tạo metadata | ❌ |
| 2 | `02_run_ppocr.py` | PP-OCRv6 detect + recognize | ✅ |
| 3 | `03_build_vlm_jobs.py` | Risk scoring + grouping + crop | ❌ |
| 4 | `04_run_vlm_correction.py` | VLM sửa lỗi chính tả | ✅ |
| 5 | `05_merge_features.py` | Gộp raw + corrected OCR | ❌ |
| 6 | `06_build_es_documents.py` | Tạo JSONL cho Elasticsearch | ❌ |

> ⚠️ **Lưu ý xung đột CUDA:** PaddleOCR và VLM (transformers + bitsandbytes) có thể xung đột thư viện CUDA. Nếu gặp lỗi, hãy **Restart Runtime** giữa Bước 2 và Bước 4.

---
## 0. Kiểm tra GPU & Clone code

In [ ]:
# Kiểm tra GPU khả dụng
!nvidia-smi

In [ ]:
# Clone code từ GitHub repo AIC-Test, nhánh Khoa
!git clone -b Khoa https://github.com/KwanFam26022005/AIC-Test.git /content/AIC-Test

# Đường dẫn gốc tới pipeline
PIPELINE_DIR = "/content/AIC-Test/data_processing/OCR/ocr_vlm_pipeline"
print(f"Pipeline directory: {PIPELINE_DIR}")

In [ ]:
# Cài đặt các thư viện cơ bản (không GPU-specific)
!pip install -q pandas pyarrow Pillow PyYAML tqdm

---
## 0.1. Chuẩn bị dữ liệu frames

Bạn cần upload hoặc mount thư mục chứa frames `.jpg` của video cần xử lý.

**Tùy chọn A:** Upload thủ công lên Colab (phù hợp demo nhỏ ~vài trăm frames).

**Tùy chọn B:** Mount Google Drive chứa dữ liệu (phù hợp scale lớn).

In [ ]:
# === TÙY CHỌN A: Sử dụng dữ liệu mẫu nhỏ để test ===
# Tạo thư mục frames mẫu (thay bằng ảnh thật của bạn)
import os
FRAMES_DIR = "/content/frames/L25_V001"
os.makedirs(FRAMES_DIR, exist_ok=True)

# Nếu bạn đã có frames, upload vào thư mục trên
# hoặc copy từ Google Drive
print(f"Frames directory: {FRAMES_DIR}")
print(f"Số frames hiện có: {len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])}")

In [ ]:
# === TÙY CHỌN B: Mount Google Drive ===
# Bỏ comment các dòng dưới nếu frames nằm trong Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# FRAMES_DIR = "/content/drive/MyDrive/AIC2026/frames/L25_V001"
# print(f"Frames directory: {FRAMES_DIR}")
# print(f"Số frames: {len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])}")

In [ ]:
# Tạo config Colab runtime (ghi đè đường dẫn phù hợp)
import yaml, os

VIDEO_ID = "L25_V001"  # <-- thay đổi nếu cần
OUTPUT_DIR = f"/content/outputs/{VIDEO_ID}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

colab_config = {
    "extends": "default.yaml",
    "project": {
        "video_id": VIDEO_ID,
        "frames_dir": FRAMES_DIR,
        "output_dir": OUTPUT_DIR,
        "resume": True,
    },
    "runtime": {
        "device": "cuda:0",
        "num_workers_io": 2,
    },
    "vlm": {
        "cache_path": f"{OUTPUT_DIR}/vlm_cache.sqlite",
    },
}

config_path = f"{PIPELINE_DIR}/configs/colab_runtime.yaml"
with open(config_path, "w") as f:
    yaml.dump(colab_config, f, default_flow_style=False)

print(f"Config saved to: {config_path}")
!cat {config_path}

---
## Bước 1: Tạo Frame Registry

Quét thư mục frames, đọc metadata (kích thước, dung lượng file), tạo danh sách tất cả frame.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/01_build_frame_registry.py --config configs/colab_runtime.yaml

In [ ]:
# === Xem kết quả Bước 1 ===
import pandas as pd

registry = pd.read_parquet(f"{OUTPUT_DIR}/frame_registry.parquet")
print(f"Tổng số frames: {len(registry)}")
print(f"Trạng thái: {registry['source_status'].value_counts().to_dict()}")
print(f"\nKích thước frames (WxH): {registry.iloc[0]['width']}x{registry.iloc[0]['height']}" if len(registry) > 0 else "Không có frame nào")
registry.head(10)

---
## Bước 2: Chạy PP-OCRv6

Sử dụng PP-OCRv6 để detect và recognize text trên tất cả frames.

> ⚠️ Bước này cần cài đặt PaddleOCR. Nếu bạn đã chạy xong bước này, có thể **Restart Runtime** rồi nhảy sang Bước 3.

In [ ]:
# Cài đặt PaddlePaddle GPU + PaddleOCR
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr>=3.0.0

In [ ]:
# Kiểm tra PaddlePaddle đã cài đặt đúng GPU
import paddle
print(f"PaddlePaddle version: {paddle.__version__}")
print(f"CUDA available: {paddle.device.is_compiled_with_cuda()}")
print(f"GPU count: {paddle.device.cuda.device_count()}")

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/02_run_ppocr.py --config configs/colab_runtime.yaml

In [ ]:
# === Xem kết quả Bước 2 ===
import pandas as pd

raw = pd.read_parquet(f"{OUTPUT_DIR}/ppocr_raw.parquet")
print(f"Tổng số OCR lines: {len(raw)}")
print(f"Số frames có text: {raw['frame_id'].nunique()}")
print(f"\nPhân bổ confidence:")
print(f"  Mean: {raw['confidence'].mean():.3f}")
print(f"  Min:  {raw['confidence'].min():.3f}")
print(f"  < 0.75: {(raw['confidence'] < 0.75).sum()} ({(raw['confidence'] < 0.75).mean()*100:.1f}%)")
print(f"\n--- Mẫu OCR text ---")
raw[['frame_id', 'line_idx', 'ocr_text', 'confidence']].head(15)

---
## Bước 3: Đánh giá rủi ro, Phân cụm & Tạo VLM Jobs

- Tính `risk_score` cho từng dòng OCR.
- Gom các bounding box gần nhau thành group.
- Cắt ảnh crop cho các group cần sửa lỗi.
- Lọc ra các VLM jobs (chỉ gửi group có rủi ro cao cho VLM).

> 💡 Bước này KHÔNG cần GPU, chạy thuần CPU.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/03_build_vlm_jobs.py --config configs/colab_runtime.yaml

In [ ]:
# === Xem kết quả Bước 3: Risk Scoring ===
import pandas as pd

risk = pd.read_parquet(f"{OUTPUT_DIR}/ocr_risk.parquet")
print(f"Tổng dòng OCR: {len(risk)}")
print(f"Cần VLM (need_vlm_line=True): {risk['need_vlm_line'].sum()} ({risk['need_vlm_line'].mean()*100:.1f}%)")
print(f"\nPhân bổ risk_score:")
print(f"  Mean: {risk['risk_score'].mean():.3f}")
print(f"  >= 0.45: {(risk['risk_score'] >= 0.45).sum()}")
print(f"\n--- Các dòng có risk cao nhất ---")
risk.nlargest(10, 'risk_score')[['frame_id', 'ocr_text', 'confidence', 'risk_score', 'risk_reasons']]

In [ ]:
# === Xem kết quả Bước 3: Grouping & VLM Jobs ===
groups = pd.read_parquet(f"{OUTPUT_DIR}/ocr_groups.parquet")
print(f"Tổng số groups: {len(groups)}")
print(f"Groups cần VLM: {groups['need_vlm_group'].sum()}")
print(f"\nPhân loại vùng (region_type):")
print(groups['region_type'].value_counts().to_string())

import os
crops_dir = f"{OUTPUT_DIR}/crops"
if os.path.exists(crops_dir):
    crop_files = [f for f in os.listdir(crops_dir) if f.endswith('.jpg')]
    print(f"\nSố ảnh crop đã tạo: {len(crop_files)}")

In [ ]:
# === Xem minh họa ảnh crop mẫu ===
import matplotlib.pyplot as plt
from PIL import Image
import os

crops_dir = f"{OUTPUT_DIR}/crops"
if os.path.exists(crops_dir):
    crop_files = sorted([f for f in os.listdir(crops_dir) if f.endswith('.jpg')])[:8]
    if crop_files:
        fig, axes = plt.subplots(2, 4, figsize=(16, 6))
        for idx, (ax, fname) in enumerate(zip(axes.flat, crop_files)):
            img = Image.open(os.path.join(crops_dir, fname))
            ax.imshow(img)
            ax.set_title(fname, fontsize=8)
            ax.axis('off')
        for ax in axes.flat[len(crop_files):]:
            ax.axis('off')
        plt.suptitle('Mẫu ảnh crop sẽ gửi cho VLM sửa lỗi', fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("Không có ảnh crop nào (có thể tất cả OCR lines đều confidence cao).")
else:
    print(f"Thư mục crops chưa tồn tại: {crops_dir}")

In [ ]:
# === Xem VLM Jobs ===
import os
vlm_jobs_path = f"{OUTPUT_DIR}/vlm_jobs.parquet"
if os.path.exists(vlm_jobs_path):
    jobs = pd.read_parquet(vlm_jobs_path)
    print(f"Số VLM jobs cần xử lý: {len(jobs)}")
    jobs[['frame_id', 'group_id', 'region_type', 'raw_group_text', 'max_risk_score']].head(10)
else:
    print("Chưa có vlm_jobs.parquet")

---
## ⚠️ QUAN TRỌNG: Restart Runtime nếu cần

Nếu bạn đã cài PaddlePaddle ở Bước 2 và bây giờ cần cài VLM stack (transformers + bitsandbytes), hãy:

1. **Runtime → Restart session** (hoặc Ctrl+M → .)
2. Sau khi restart, chạy lại cell **Clone code** (mục 0) và cell **Config** để khôi phục biến.
3. Sau đó tiếp tục từ Bước 4 bên dưới.

Nếu không gặp lỗi xung đột, bạn có thể bỏ qua bước restart và chạy tiếp.

In [ ]:
# === Chạy cell này SAU KHI restart runtime ===
# Khôi phục lại các biến đường dẫn

PIPELINE_DIR = "/content/AIC-Test/data_processing/OCR/ocr_vlm_pipeline"
VIDEO_ID = "L25_V001"  # <-- giữ đúng với giá trị đã dùng ở trên
FRAMES_DIR = "/content/frames/L25_V001"  # <-- giữ đúng với giá trị đã dùng ở trên
OUTPUT_DIR = f"/content/outputs/{VIDEO_ID}"

print(f"Pipeline: {PIPELINE_DIR}")
print(f"Frames:   {FRAMES_DIR}")
print(f"Output:   {OUTPUT_DIR}")

import os
for f in ['frame_registry.parquet', 'ppocr_raw.parquet', 'ocr_risk.parquet', 'ocr_groups.parquet', 'vlm_jobs.parquet']:
    path = os.path.join(OUTPUT_DIR, f)
    status = '✅' if os.path.exists(path) else '❌'
    print(f"  {status} {f}")

---
## Bước 4: Chạy VLM Correction

Cài đặt stack VLM (transformers, bitsandbytes) và chạy sửa lỗi chính tả tiếng Việt bằng Vintern-3B.

> ✅ Bước này cần GPU. Model Vintern-3B ở chế độ 4-bit chỉ dùng ~3.5-5GB VRAM, phù hợp T4 (16GB).

In [ ]:
# Cài đặt VLM dependencies
!pip install -q transformers==4.44.2 accelerate einops timm sentencepiece bitsandbytes
!pip install -q pandas pyarrow Pillow PyYAML tqdm

In [ ]:
# Kiểm tra GPU + torch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# Chạy VLM correction
# --limit N: giới hạn số lượng groups để test nhanh (bỏ tham số này để chạy hết)
%cd {PIPELINE_DIR}
!python scripts/04_run_vlm_correction.py --config configs/colab_runtime.yaml --limit 10

In [ ]:
# === Xem kết quả Bước 4 ===
import pandas as pd

corrected_path = f"{OUTPUT_DIR}/vlm_corrected.parquet"
import os
if os.path.exists(corrected_path):
    corrected = pd.read_parquet(corrected_path)
    print(f"Tổng dòng đã sửa: {len(corrected)}")
    print(f"Parse status: {corrected['parse_status'].value_counts().to_dict()}")
    
    changed = corrected[corrected['raw_text'] != corrected['corrected_text']]
    print(f"\nSố dòng text thay đổi: {len(changed)} ({len(changed)/max(1,len(corrected))*100:.1f}%)")
    
    print(f"\n--- So sánh raw vs corrected ---")
    for _, row in changed.head(10).iterrows():
        print(f"  [{row['frame_id']}] raw:       {row['raw_text']}")
        print(f"  [{row['frame_id']}] corrected: {row['corrected_text']}")
        print()
else:
    print("Chưa có vlm_corrected.parquet — hãy chạy Bước 4 trước.")

---
## Bước 5: Gộp kết quả (Merge)

Gộp dữ liệu OCR thô với dữ liệu đã được VLM sửa lỗi, tạo bảng tổng hợp cấp frame.

> 💡 Bước này KHÔNG cần GPU.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/05_merge_features.py --config configs/colab_runtime.yaml

In [ ]:
# === Xem kết quả Bước 5: Chi tiết từng dòng ===
import pandas as pd

merged = pd.read_parquet(f"{OUTPUT_DIR}/ocr_merged_lines.parquet")
print(f"Tổng dòng merged: {len(merged)}")
print(f"Dòng có VLM correction: {merged['vlm_corrected'].sum()}")
print(f"Dòng text thay đổi: {merged['text_changed'].sum()}")

merged[['frame_id', 'line_idx', 'ocr_text', 'corrected_text', 'normalized_text', 'vlm_corrected', 'text_changed']].head(15)

In [ ]:
# === Xem kết quả Bước 5: Tóm tắt cấp frame ===
summary = pd.read_parquet(f"{OUTPUT_DIR}/ocr_frame_summary.parquet")
print(f"Tổng frames có OCR: {len(summary)}")
print(f"Frames có VLM correction: {summary['vlm_corrected'].sum()}")

summary[['frame_id', 'line_count', 'avg_confidence', 'vlm_corrected', 'ocr_corrected_text']].head(10)

---
## Bước 6: Tạo tài liệu Elasticsearch (JSONL)

Chuyển đổi dữ liệu frame-level thành file JSONL sẵn sàng đẩy vào Elasticsearch.

> 💡 Bước này KHÔNG cần GPU.

In [ ]:
%cd {PIPELINE_DIR}
!python scripts/06_build_es_documents.py --config configs/colab_runtime.yaml

In [ ]:
# === Xem kết quả Bước 6 ===
import json

es_path = f"{OUTPUT_DIR}/es_documents.jsonl"
import os
if os.path.exists(es_path):
    with open(es_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"Tổng số documents: {len(lines)}")
    
    # Hiển thị document đầu tiên
    if lines:
        doc = json.loads(lines[0])
        print(f"\n--- Document mẫu ---")
        print(json.dumps(doc, ensure_ascii=False, indent=2)[:2000])
else:
    print("Chưa có es_documents.jsonl")

---
## Tải kết quả về máy Local

Sau khi hoàn tất, bạn có thể download các file kết quả về máy cá nhân để chạy Bước 7 & 8 (Elasticsearch) ở local.

In [ ]:
# Nén toàn bộ kết quả thành file zip để download
import shutil
zip_path = shutil.make_archive(f'/content/pipeline_output_{VIDEO_ID}', 'zip', OUTPUT_DIR)
print(f"File zip: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")

# Download tự động (Colab)
from google.colab import files
files.download(zip_path)

---
## Tóm tắt các file output

| File | Mô tả |
|------|--------|
| `frame_registry.parquet` | Danh sách metadata toàn bộ frames |
| `ppocr_raw.parquet` | Kết quả OCR thô (line-level) |
| `ocr_risk.parquet` | Điểm rủi ro từng dòng OCR |
| `ocr_groups.parquet` | Kết quả gom nhóm + phân loại vùng |
| `vlm_jobs.parquet` | Danh sách công việc gửi cho VLM |
| `vlm_corrected.parquet` | Kết quả sửa lỗi từ VLM |
| `ocr_merged_lines.parquet` | Dữ liệu gộp raw + corrected (line-level) |
| `ocr_frame_summary.parquet` | Tóm tắt OCR cấp frame |
| `es_documents.jsonl` | Tài liệu sẵn sàng đẩy vào Elasticsearch |
| `crops/` | Ảnh crop gửi cho VLM |
| `vlm_cache.sqlite` | Cache kết quả VLM (tránh chạy lại) |

In [ ]:
# === Tổng kết kết quả toàn bộ pipeline ===
import os

print(f"=" * 60)
print(f"  TỔNG KẾT PIPELINE - {VIDEO_ID}")
print(f"=" * 60)

files_info = [
    ('frame_registry.parquet', 'Frame Registry'),
    ('ppocr_raw.parquet', 'PP-OCR Raw'),
    ('ocr_risk.parquet', 'Risk Scoring'),
    ('ocr_groups.parquet', 'OCR Groups'),
    ('vlm_jobs.parquet', 'VLM Jobs'),
    ('vlm_corrected.parquet', 'VLM Corrected'),
    ('ocr_merged_lines.parquet', 'Merged Lines'),
    ('ocr_frame_summary.parquet', 'Frame Summary'),
    ('es_documents.jsonl', 'ES Documents'),
    ('vlm_cache.sqlite', 'VLM Cache'),
]

for fname, label in files_info:
    path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✅ {label:20s} | {fname:30s} | {size_kb:8.1f} KB")
    else:
        print(f"  ❌ {label:20s} | {fname:30s} | chưa tạo")

crops_dir = os.path.join(OUTPUT_DIR, 'crops')
if os.path.exists(crops_dir):
    n_crops = len([f for f in os.listdir(crops_dir) if f.endswith('.jpg')])
    print(f"  📁 Crops directory   | crops/                         | {n_crops} files")

print(f"=" * 60)